In [103]:
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

In [104]:
path = "../data/raw/"

all_csv = os.listdir(path)
df_list = []
for file in all_csv:
    path_file = os.path.join(path,file)
    df_list.append(pd.read_csv(path_file))

df = pd.concat(df_list)
df.sort_values(by='respondent_id',inplace=True,ignore_index=True)
df.index += 1
df.head()

,age_group,gender,college_year,monthly_budget_range,monthly_budget_exact,income_source,primary_upi_app,weekly_tx_range,weekly_tx_count,pct_unplanned,...,regret_other_text,post_regret_action,regret_intensity,hidden_purchase,tracks_expenses,ran_out_of_money,perceives_upi_risky,upi_usage_reason,regret_description,respondent_id
1,22-25,Male,Ug,₹6000+ (Force fully leaving in the college hos...,18000.0,100% Parents Money,PhonePe,20+ (You are the reason companies are profitab...,36.0,20-40% (Understandable),...,NaN,Accept it or move on (aaj tak nahi huva),3,No (we don't believe),"No, I need some surprise in life","Yes, regularly",Yes,I feel comfortable using UPI,Marrow course,1000
2,18-21,Male,Ug,₹1000 - ₹3000 (Mess food is not enough for me),3500.0,100% Parents Money,PhonePe,20+ (You are the reason companies are profitab...,40.0,20-40% (Understandable),...,NaN,Accept it or move on (aaj tak nahi huva),3,No (we don't believe),"No, I need some surprise in life","Yes, Occasionally",Yes,I feel comfortable using UPI,NaN,1001
3,18-21,Male,Ug,₹1000 - ₹3000 (Mess food is not enough for me),2000.0,100% Parents Money,PhonePe,20+ (You are the reason companies are profitab...,21.0,40-60% (Mostly it on the food),...,NaN,Accept it or move on (aaj tak nahi huva),1,Maybe (don't want to accept it),"I try, but give up by one or two week (Relatable)",Rarely,Yes,I feel comfortable using UPI,NaN,1002
4,18-21,Male,Ug,₹3000 - ₹6000 (I need to go out of this campus),NaN,100% Parents Money,Google Pay (GPay),15-20 (Loves eating too much),NaN,60-80% (Probably you love everything you see o...,...,NaN,Accept it or move on (aaj tak nahi huva),1,Maybe (don't want to accept it),"No, I need some surprise in life",Rarely,Yes,I feel comfortable using UPI,NaN,1003
5,Under 18,Female,Ug,₹3000 - ₹6000 (I need to go out of this campus),NaN,100% Parents Money,PhonePe,8-15 (The apps know your face by now),NaN,20-40% (Understandable),...,NaN,Accept it or move on (aaj tak nahi huva),2,NaN,"I try, but give up by one or two week (Relatable)","Yes, Occasionally",Yes,I feel comfortable using UPI,NaN,1004


In [105]:
df_clean = df.iloc[:,[31,0,1,2]]

In [106]:
df['monthly_budget_range'].unique()

array(['₹6000+ (Force fully leaving in the college hostel)',
       '₹1000 - ₹3000 (Mess food is not enough for me)',
       '₹3000 - ₹6000 (I need to go out of this campus)',
       'Less than ₹1000 (Enjoy hostel Wi-Fi and mess food)',
       '₹6000+ (Force fully living in the college hostel)'], dtype=object)

In [107]:
df_clean['monthly_budget_range'] = df['monthly_budget_range'].replace({
    '₹6000+ (Force fully leaving in the college hostel)' : 'Rs6000+',
       '₹1000 - ₹3000 (Mess food is not enough for me)' : 'Rs1000 - Rs3000',
       '₹3000 - ₹6000 (I need to go out of this campus)' : 'Rs3000 - Rs6000',
       'Less than ₹1000 (Enjoy hostel Wi-Fi and mess food)' : '< Rs1000',
       '₹6000+ (Force fully living in the college hostel)' : 'Rs6000+'
})

income_map = {
    'Rs6000+' : 7000,
    'Rs1000 - Rs3000' : 2000,
    'Rs3000 - Rs6000' : 4500,
    '< Rs1000' : 750
}

df_clean['avg_monthly_budget'] = df['monthly_budget_exact'].fillna(df_clean['monthly_budget_range'].map(income_map))

In [108]:
df['income_source'].value_counts()

income_source
100% Parents Money                                  86
Mix of parents + I earn something on the side       10
I have an internship / part-time job                 5
I run a side hustle / freelance / small business     4
Name: count, dtype: int64

In [109]:
df_clean['income_source'] = df['income_source']

In [110]:
df['primary_upi_app'].value_counts()

primary_upi_app
Google Pay (GPay)               47
PhonePe                         28
Paytm                           14
Bhim UPI                         5
Navi UPI                         5
Cred                             2
Amazon Pay                       1
Google pay as well as Paytm      1
PayZapp                          1
Fampay                           1
Name: count, dtype: int64

In [111]:
df_clean['primary_upi_app'] = df['primary_upi_app'].replace({'Google pay as well as Paytm ' : 'Paytm'})
df_clean['primary_upi_app'].value_counts()

primary_upi_app
Google Pay (GPay)    47
PhonePe              28
Paytm                15
Bhim UPI              5
Navi UPI              5
Cred                  2
Amazon Pay            1
PayZapp               1
Fampay                1
Name: count, dtype: int64

In [112]:
df['weekly_tx_range'].value_counts()

weekly_tx_range
4-7 (Okay this is reasonable )                        40
8-15 (The apps know your face by now)                 29
1-3 (No comments)                                     16
15+ (You are the reason companies are profitable )    16
20+ (You are the reason companies are profitable )     3
15-20 (Loves eating too much)                          1
Name: count, dtype: int64

In [113]:
df_clean['weekly_tx_range'] = df['weekly_tx_range'].replace({
    '1-3 (No comments)' : '1-3',
    '4-7 (Okay this is reasonable )' : '4-7',
    '8-15 (The apps know your face by now)' : '8-15',
    '15+ (You are the reason companies are profitable )' : '15+',
    '20+ (You are the reason companies are profitable )' : '15+',
    '15-20 (Loves eating too much)' : '15+'
})

upi_tx_map = {
    '1-3' : 2,
    '4-7' : 5,
    '8-15': 12,
    '15+': 18
}

df_clean['avg_weekly_tx'] = df['weekly_tx_count'].fillna(df_clean['weekly_tx_range'].map(upi_tx_map))

In [114]:
df['pct_unplanned'].value_counts()

pct_unplanned
40-60% (Mostly it on the food)                           39
20-40% (Understandable)                                  34
Less than 20% (its a lie)                                23
60-80% (Probably you love everything you see online)      5
More than 80% (You need help for that fill this form)     4
Name: count, dtype: int64

In [115]:
df_clean['pct_unplanned'] = df['pct_unplanned'].replace({
    '40-60% (Mostly it on the food)' : '40-60%',
    '20-40% (Understandable)' : '20-40%',
    'Less than 20% (its a lie)' : '<20%',
    '60-80% (Probably you love everything you see online)' : '60-80%',
    'More than 80% (You need help for that fill this form)' : '80%+',
})

df_clean['pct_unplanned_avg'] = df_clean['pct_unplanned'].replace({
    '40-60%' : 50,
    '20-40%' : 30,
    '<20%' : 10,
    '60-80%' : 70,
    '80%+' : 90
})

In [116]:
df['balance_check_habit'].value_counts()

balance_check_habit
Sometimes (When I remember I'm poor)          47
Always (I am disciplined and boring)          20
Never (Pls check it for yourself)             18
Rarely (Assuming account is full of money)    13
I check, feel sad, and buy anyway              7
Name: count, dtype: int64

In [117]:
df_clean['balance_check_habit'] = df['balance_check_habit'].replace({
    "Sometimes (When I remember I'm poor)" : 'Sometimes',
    'Always (I am disciplined and boring)' : 'Always',
    'Never (Pls check it for yourself)' : 'Never',
    'Rarely (Assuming account is full of money)' : 'Rarely',
    'I check, feel sad, and buy anyway' : 'Rarely',
})

blance_habit_map = {
    'Sometimes' : 2,
    'Always' : 3,
    'Never' : 0,
    'Rarely' : 1
}

df_clean['balance_check_habit'] = df_clean['balance_check_habit'].map(blance_habit_map)

In [118]:
df_clean['flag_morning'] = df['impulse_time'].str.contains("Morning", case=False).astype(int)
df_clean['flag_afternoon'] = df['impulse_time'].str.contains("Afternoon", case=False).astype(int)
df_clean['flag_evening'] = df['impulse_time'].str.contains("Evening", case=False).astype(int)
df_clean['flag_latenight'] = df['impulse_time'].str.contains("Late Night", case=False).astype(int)
df_clean['flag_postmidnight'] = df['impulse_time'].str.contains("Post Midnight", case=False).astype(int)

In [119]:
df["regret_frequency"].value_counts()

regret_frequency
1-2 times               51
Never                   28
3-5 times               16
10+ (Lost the count)     5
5-10 times               4
I have lost count        1
Name: count, dtype: int64

In [120]:
df_clean['regret_frequency'] = df['regret_frequency'].str.replace(r"\(.*\)", "", regex=True).str.strip()

df_clean['regret_frequency'] = df_clean['regret_frequency'].replace({
    "I have lost count": "10+",
    "10+": "10+"
})

regret_map = {
    "Never": 0,
    "1-2 times": 1,
    "3-5 times": 2,
    "5-10 times": 3,
    "10+": 4
}

df_clean['regret_frequency'] = df_clean['regret_frequency'].map(regret_map)

In [121]:
df_clean['cat_food_delivery'] = df['regret_categories'].str.contains("food", case=False).astype(int)
df_clean['cat_grocery'] = df['regret_categories'].str.contains("grocery|blinkit|instamart", case=False).astype(int)
df_clean['cat_online_shopping'] = df['regret_categories'].str.contains("online", case=False).astype(int)
df_clean['cat_subscriptions'] = df['regret_categories'].str.contains("subscription", case=False).astype(int)
df_clean['cat_gaming'] = df['regret_categories'].str.contains("gaming", case=False).astype(int)
df_clean['cat_gadgets'] = df['regret_categories'].str.contains("gadget", case=False).astype(int)
df_clean['cat_offline_cafe'] = df['regret_categories'].str.contains("cafe|restaurant", case=False).astype(int)
df_clean['cat_other'] = df['regret_categories'].str.contains("other", case=False).astype(int)

In [122]:
df_clean['post_regret_action'] = df['post_regret_action'].str.replace(r"\(.*\)", "", regex=True).str.strip()

In [123]:
df_clean['hidden_purchase'] = df['hidden_purchase'].str.replace(r"\(.*\)", "", regex=True).str.strip()

df_clean['hidden_purchase'] = df_clean['hidden_purchase'].fillna("No")

hidden_map = {
    "No": 0,
    "Maybe": 1,
    "Yes": 2
}

df_clean['hidden_purchase'] = df_clean['hidden_purchase'].map(hidden_map)

In [124]:
df_clean['tracks_expenses'] = df['tracks_expenses'].str.replace(r"\(.*\)", "", regex=True).str.strip()

df_clean['tracks_expenses'] = df_clean['tracks_expenses'].replace({
    "I am scared of what I'll find": "No",
    "I try, but give up by one or two week" : "Tries but quits",
    "No, I need some surprise in life" : "No"
})

tracks_map = {
    "No": 0,
    "Tries but quits": 1,
    "Yes": 2
}

df_clean['tracks_expenses'] = df_clean['tracks_expenses'].map(tracks_map)

In [125]:
ranout_map = {
    "Never (teach us your ways)": 0,
    "Rarely": 1,
    "Yes, Occasionally": 2,
    "Yes, regularly": 3
}

df_clean['ran_out_of_money'] = df['ran_out_of_money'].replace(ranout_map)

In [126]:
df_clean['perceives_upi_risky'] = df['perceives_upi_risky'].replace({
    "Yes": 1,
    "No": 0,
    "I never thought about this": 0
})

In [127]:
df['upi_usage_reason'] = df['upi_usage_reason'].str.lower()

df_clean['upi_usage_reason'] = "Convenience"

df_clean.loc[df['upi_usage_reason'].str.contains("habit|comfortable"), 'upi_usage_reason'] = "Comfort"
df_clean.loc[df['upi_usage_reason'].str.contains("safe|security|lose cash"), 'upi_usage_reason'] = "Safety"
df_clean.loc[df['upi_usage_reason'].str.contains("shop|merchant"), 'upi_usage_reason'] = "Merchant preference"

In [128]:
df_clean['regret_intensity'] = df['regret_intensity']
df_clean['high_regret'] = (df['regret_intensity'] > 5).astype(int)

In [129]:
df_clean["regret_description"] = df["regret_description"]

In [130]:
trigger = ['trigger_boredom','trigger_fomo','trigger_latenight','trigger_cashback','trigger_stress_relief','trigger_scarcity_notif','trigger_cart_abandon','trigger_exam_season']
df_clean[trigger] = df[trigger]

In [131]:
df_clean.columns

Index(['respondent_id', 'age_group', 'gender', 'college_year',
       'monthly_budget_range', 'avg_monthly_budget', 'income_source',
       'primary_upi_app', 'weekly_tx_range', 'avg_weekly_tx', 'pct_unplanned',
       'pct_unplanned_avg', 'balance_check_habit', 'flag_morning',
       'flag_afternoon', 'flag_evening', 'flag_latenight', 'flag_postmidnight',
       'regret_frequency', 'cat_food_delivery', 'cat_grocery',
       'cat_online_shopping', 'cat_subscriptions', 'cat_gaming', 'cat_gadgets',
       'cat_offline_cafe', 'cat_other', 'post_regret_action',
       'hidden_purchase', 'tracks_expenses', 'ran_out_of_money',
       'perceives_upi_risky', 'upi_usage_reason', 'regret_intensity',
       'high_regret', 'regret_description', 'trigger_boredom', 'trigger_fomo',
       'trigger_latenight', 'trigger_cashback', 'trigger_stress_relief',
       'trigger_scarcity_notif', 'trigger_cart_abandon',
       'trigger_exam_season'],
      dtype='object')

In [132]:
new_order = [
    #Demographics
    'respondent_id', 'age_group', 'gender', 'college_year',
    #Financial profile
    'monthly_budget_range', 'avg_monthly_budget', 'income_source',
    #UPI / payment behavior
    'primary_upi_app', 'perceives_upi_risky', 'upi_usage_reason',
    #Transaction behavior
    'weekly_tx_range', 'avg_weekly_tx',
    'pct_unplanned', 'pct_unplanned_avg',
    'balance_check_habit', 'tracks_expenses',
    'ran_out_of_money',
    #Impulse triggers (time-based flags)
    'flag_morning', 'flag_afternoon', 'flag_evening',
    'flag_latenight', 'flag_postmidnight',
    #Impulse triggers (behavioral)
    'trigger_boredom', 'trigger_fomo', 'trigger_latenight',
    'trigger_cashback', 'trigger_stress_relief',
    'trigger_scarcity_notif', 'trigger_cart_abandon',
    'trigger_exam_season',
    #Regret behavior
    'regret_frequency',
    'cat_food_delivery', 'cat_grocery', 'cat_online_shopping',
    'cat_subscriptions', 'cat_gaming', 'cat_gadgets',
    'cat_offline_cafe', 'cat_other',
    'post_regret_action', 'hidden_purchase',
    #Target
    'regret_intensity', 'high_regret',
    #Text
    'regret_description'
]

In [133]:
df_clean = df_clean[new_order]

In [134]:
df_clean["perceives_upi_risky"].value_counts()

perceives_upi_risky
1                            82
0                            16
I never taught about this     7
Name: count, dtype: int64

In [135]:
x = df_clean["perceives_upi_risky"].unique()[1]
df_clean["perceives_upi_risky"] = df_clean["perceives_upi_risky"].replace({
    x : 0
})
df_clean["perceives_upi_risky"].value_counts()

perceives_upi_risky
1    82
0    23
Name: count, dtype: int64

In [136]:
df.to_csv("../data/processed/combined.csv", index=False, sep=',')
df_clean.to_csv("../data/processed/combined_clean.csv", index=False, sep=',')